# Corrected FADC3D encoder — seed 42 — NO DEEP SUPERVISION

Controlled ablation against the deep-supervision run
(`kaggle_train_fadc3d_correct_encoder_s42.ipynb`, best formal Dice 0.7068).

**Only intended difference vs. the DS run:** deep supervision is disabled.
Every other setting (model, seed, epochs, batch, patch, LR, warmup, workers,
k_att schedule, validation protocol, preprocessed cache, augmentation,
Dice+CE loss, attention-diversity weight, position attention) is preserved.

Output directory is separate (`..._nods_s42`) so this run cannot overwrite
or resume from the DS artifacts.

Fresh training only: this notebook does not read any checkpoint from the
DS run and refuses to auto-resume from a stale file in its own output dir.


In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────
# Controlled ablation: identical to kaggle_train_fadc3d_correct_encoder_s42
# except DEEP_SUPERVISION=False. Do NOT modify any other hyperparameter here.
SEED                = 42
GIT_BRANCH          = "feature/fadc3d-correct"

DATA_ROOT              = "/kaggle/input/datasets/bharathvemurik/mama-mia-preprocessed-cache-2ch"
PREPROCESSED_CACHE_DIR = DATA_ROOT
CODE_DIR               = "/kaggle/working/FADC-3D"

# Separate output directories so this notebook cannot touch DS artifacts.
OUTPUT_DIR_SMOKE  = "/kaggle/working/outputs/fadc3d_correct_encoder_nods_smoke"
OUTPUT_DIR_FULL   = "/kaggle/working/outputs/fadc3d_correct_encoder_nods_s42"

# Full-training hyperparameters (UNCHANGED vs. DS run — do NOT modify).
EPOCHS         = 100
BATCH_SIZE     = 2
NUM_WORKERS    = 4
PATCH_SIZE     = [128, 128, 64]
LEARNING_RATE  = 1e-4
WARMUP_EPOCHS  = 5

# ── THE ONE INTENDED DIFFERENCE VS. DS RUN ─────────────────────────────
DEEP_SUPERVISION = False

# k_att schedule (UNCHANGED).
K_ATT_TEMP_START    = 2.0
K_ATT_TEMP_END      = 1.0
K_ATT_ANNEAL_EPOCHS = 60

# Attention diversity aux DISABLED (UNCHANGED).
ATTN_DIVERSITY_WEIGHT = 0.0

# Position attention DISABLED (UNCHANGED — default; --use_position_att is
# a store_true flag on the training script and is NOT passed by this notebook).
USE_POSITION_ATT = False

# Smoke-test parameters.
SMOKE_PATCH_SIZE = [48, 48, 24]

MODEL_NAME = "unet3d_fadc_encoder_correct"

# ── VALIDATION SCHEDULE (formal only) ──────────────────────────────────
# One protocol: all validation cases, sliding-window at overlap=0.5.
# Runs every VAL_EVERY epochs. Only formal validation updates
# best_model.pth. No fast/proxy variant.
VAL_EVERY           = 20     # formal validation every N epochs
VAL_OVERLAP         = 0.5    # canonical evaluation overlap
VAL_SW_BATCH_SIZE   = 4
CHECKPOINT_EVERY    = 10     # also save named checkpoint_epochNNN.pth

# ── FRESH TRAINING ONLY ────────────────────────────────────────────────
# The DS notebook exposed a RESUME_FROM knob. This ablation is fresh-only
# by contract: we never pass --resume to the training script, and the
# preflight cell aborts if a stale checkpoint is found under OUTPUT_DIR_FULL.

print(f"SEED                : {SEED}")
print(f"BRANCH              : {GIT_BRANCH}")
print(f"MODEL_NAME          : {MODEL_NAME}")
print(f"OUTPUT_DIR_FULL     : {OUTPUT_DIR_FULL}")
print(f"OUTPUT_DIR_SMOKE    : {OUTPUT_DIR_SMOKE}")
print(f"DEEP_SUPERVISION    : {DEEP_SUPERVISION}    <-- ABLATION KNOB")
print(f"PATCH_SIZE (train)  : {PATCH_SIZE}")
print(f"PATCH_SIZE (smoke)  : {SMOKE_PATCH_SIZE}")
print(f"EPOCHS              : {EPOCHS}")
print(f"BATCH_SIZE          : {BATCH_SIZE}")
print()
print("VALIDATION SCHEDULE (formal only)")
print(f"  validation cases    : 306 (all)")
print(f"  formal overlap      : {VAL_OVERLAP}")
print(f"  validation frequency: every {VAL_EVERY} epochs")
print(f"  sw_batch_size       : {VAL_SW_BATCH_SIZE}")
print(f"  checkpoint_every    : {CHECKPOINT_EVERY}")
print()
print(f"k_att T             : {K_ATT_TEMP_START} -> {K_ATT_TEMP_END} over {K_ATT_ANNEAL_EPOCHS} ep")
print(f"attn_diversity_wt   : {ATTN_DIVERSITY_WEIGHT}  (disabled)")
print(f"use_position_att    : {USE_POSITION_ATT}   (disabled)")
print(f"RESUME              : disabled — fresh training only")
print()
print("WARNING: FORMAL validation over all 306 cases at overlap=0.5 may take")
print("         SEVERAL HOURS on Kaggle GPUs. Plan runs accordingly.")


In [ ]:
# ── PREFLIGHT — no-DS ablation contract ─────────────────────────────────
# Static assertions that must hold before ANY expensive cell runs. If any
# assertion fires, fix the config and re-run from the top — do not skip.
import os, shlex, sys

# 1) DEEP_SUPERVISION must be False.
assert DEEP_SUPERVISION is False, \
    f"PREFLIGHT: DEEP_SUPERVISION must be False for this ablation (got {DEEP_SUPERVISION!r})."

# 2) Output directory must be the nods variant (defence-in-depth vs. accidental
#    edits pointing at the DS run's output tree).
assert "nods" in OUTPUT_DIR_FULL, \
    f"PREFLIGHT: OUTPUT_DIR_FULL must contain 'nods' (got {OUTPUT_DIR_FULL!r})."
assert "nods" in OUTPUT_DIR_SMOKE, \
    f"PREFLIGHT: OUTPUT_DIR_SMOKE must contain 'nods' (got {OUTPUT_DIR_SMOKE!r})."
assert OUTPUT_DIR_FULL != "/kaggle/working/outputs/fadc3d_correct_encoder_s42", \
    "PREFLIGHT: OUTPUT_DIR_FULL points at the DS run's output directory. Refusing."

# 3) Model name must be the encoder-placement corrected model.
assert MODEL_NAME == "unet3d_fadc_encoder_correct", \
    f"PREFLIGHT: MODEL_NAME must be 'unet3d_fadc_encoder_correct' (got {MODEL_NAME!r})."

# 4) No resume checkpoint must be selected. RESUME_FROM was deliberately
#    removed from the CONFIG cell for this ablation; if a future edit
#    reintroduces it, the next block will still catch it because we never
#    pass --resume in the training command.
assert "RESUME_FROM" not in globals() or not globals().get("RESUME_FROM"), \
    "PREFLIGHT: RESUME_FROM must be unset/empty; this ablation is fresh-only."

# 5) Training command sanity — reconstruct the same argv the training cell
#    will build and confirm neither --deep_supervision nor --resume appears.
_training_cmd_preview = [
    sys.executable, "-u", "training/train_centralized_correct.py",
    "--model",              MODEL_NAME,
    "--data_root",          DATA_ROOT,
    "--output_dir",         OUTPUT_DIR_FULL,
    "--epochs",             str(EPOCHS),
    "--batch_size",         str(BATCH_SIZE),
    "--num_workers",        str(NUM_WORKERS),
    "--patch_size",         str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
    "--lr",                 str(LEARNING_RATE),
    "--warmup_epochs",      str(WARMUP_EPOCHS),
    "--seed",               str(SEED),
    "--k_att_temp_start",   str(K_ATT_TEMP_START),
    "--k_att_temp_end",     str(K_ATT_TEMP_END),
    "--k_att_anneal_epochs",str(K_ATT_ANNEAL_EPOCHS),
    "--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR,
    "--val_every",          str(VAL_EVERY),
    "--val_overlap",        str(VAL_OVERLAP),
    "--val_sw_batch_size",  str(VAL_SW_BATCH_SIZE),
    "--checkpoint_every",   str(CHECKPOINT_EVERY),
]
assert "--deep_supervision" not in _training_cmd_preview, \
    "PREFLIGHT: training command must NOT contain --deep_supervision."
assert "--resume" not in _training_cmd_preview, \
    "PREFLIGHT: training command must NOT contain --resume."

# 6) Validation overlap contract — formal protocol is 0.5. Fast/proxy would
#    be 0.0 and is explicitly not part of this ablation.
assert abs(VAL_OVERLAP - 0.5) < 1e-9, \
    f"PREFLIGHT: VAL_OVERLAP must be 0.5 (got {VAL_OVERLAP!r})."

# 7) Train/validation SET overlap check — the preprocessed cache carries
#    disjoint 'train' and 'val' subdirectories. Assert their patient-id
#    intersection is empty (protects against data leakage independent of
#    the DS/no-DS knob). If the cache is not mounted (running notebook
#    locally), skip with a warning — Kaggle will exercise this.
_train_dir = os.path.join(PREPROCESSED_CACHE_DIR, "train")
_val_dir   = os.path.join(PREPROCESSED_CACHE_DIR, "val")
if os.path.isdir(_train_dir) and os.path.isdir(_val_dir):
    _train_ids = {n for n in os.listdir(_train_dir) if not n.startswith(".")}
    _val_ids   = {n for n in os.listdir(_val_dir)   if not n.startswith(".")}
    _overlap   = _train_ids & _val_ids
    assert not _overlap, \
        f"PREFLIGHT: train/val case-id overlap is non-empty (n={len(_overlap)}): " \
        f"{sorted(list(_overlap))[:5]}..."
    print(f"train/val case-id overlap : 0  (train={len(_train_ids)}, val={len(_val_ids)})")
else:
    print(f"train/val case-id overlap : SKIPPED (cache not mounted locally — Kaggle will check).")

# 8) Adaptive-conv count check — deferred to the architecture cell (needs
#    torch + model factory). The preflight only asserts the model NAME here;
#    the count-8 check runs in the pull-verifier cell below.
_expected_adaptive_convs = 8  # documented contract; enforced later by build_unet3d_fadc_correct

# 9) Guard against auto-resume from a stale checkpoint that might already
#    live under OUTPUT_DIR_FULL (e.g. from a re-run on the same Kaggle
#    session). Any of these would silently bias the ablation.
_stale = []
if os.path.isdir(OUTPUT_DIR_FULL):
    for name in ("best_model.pth", "last_checkpoint.pth"):
        p = os.path.join(OUTPUT_DIR_FULL, name)
        if os.path.exists(p):
            _stale.append(p)
    for name in os.listdir(OUTPUT_DIR_FULL):
        if name.startswith("checkpoint_epoch") and name.endswith(".pth"):
            _stale.append(os.path.join(OUTPUT_DIR_FULL, name))
if _stale:
    raise SystemExit(
        "PREFLIGHT: OUTPUT_DIR_FULL already contains checkpoint files:\n  "
        + "\n  ".join(_stale)
        + "\n\nThis ablation is fresh-only. Delete these files (or point "
          "OUTPUT_DIR_FULL at a clean directory) and re-run the CONFIG cell."
    )

# 10) Emit a one-line summary the training cell can grep for in reviews.
print()
print("PREFLIGHT OK")
print(f"  DEEP_SUPERVISION       : {DEEP_SUPERVISION}")
print(f"  OUTPUT_DIR_FULL        : {OUTPUT_DIR_FULL}")
print(f"  MODEL_NAME             : {MODEL_NAME}")
print(f"  adaptive_convs contract: {_expected_adaptive_convs}")
print(f"  VAL_OVERLAP            : {VAL_OVERLAP}")
print(f"  --deep_supervision     : NOT in training command")
print(f"  --resume               : NOT in training command")
print(f"  stale checkpoints      : none under OUTPUT_DIR_FULL")


In [ ]:
# ── 1. INSTALL DEPS + REQUIRE CUDA ─────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "monai",
                "--upgrade-strategy", "only-if-needed", "-q"], check=True)

import torch
print(f"PyTorch        : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise SystemExit(
        "CUDA is not available on this session. This notebook is designed "
        "to run on a GPU-backed Kaggle kernel — switch the accelerator "
        "to GPU (any single-GPU option works) and re-run.\n"
        "NOTE: This training code is SINGLE-GPU and uses only cuda:0. "
        "Selecting the 'T4 x2' accelerator does NOT combine both GPUs or "
        "double the available VRAM — the second T4 sits idle. Use 'P100' "
        "or 'T4 x2' interchangeably; only cuda:0's VRAM budget matters."
    )
print(f"GPU (cuda:0)   : {torch.cuda.get_device_name(0)}")
p = torch.cuda.get_device_properties(0)
print(f"VRAM (cuda:0)  : {p.total_memory / 1e9:.1f} GB")
print("NOTE: this notebook uses only cuda:0. 'T4 x2' is NOT multi-GPU here.")


In [ ]:
# ── 2. CLONE / CHECKOUT feature/fadc3d-correct (subprocess, check=True) ─
import os, sys, subprocess

def _run(cmd, cwd=None):
    """subprocess.run wrapper that surfaces stderr and check=True."""
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        sys.stdout.write(r.stdout)
        sys.stderr.write(r.stderr)
        raise SystemExit(f"Command failed ({r.returncode}): {' '.join(cmd)}")
    return r.stdout.strip()

if os.path.exists(CODE_DIR):
    print(f"repo present; fetching {GIT_BRANCH} ...")
    _run(["git", "-C", CODE_DIR, "fetch", "--all"])
    _run(["git", "-C", CODE_DIR, "checkout", GIT_BRANCH])
    _run(["git", "-C", CODE_DIR, "pull", "--ff-only"])
else:
    _run(["git", "clone", "-b", GIT_BRANCH,
          "https://github.com/Vemuri-BK/FADC-3D.git", CODE_DIR])

sys.path.insert(0, CODE_DIR)

# Verify active branch matches GIT_BRANCH and capture the exact commit hash.
active = _run(["git", "-C", CODE_DIR, "rev-parse", "--abbrev-ref", "HEAD"])
if active != GIT_BRANCH:
    raise SystemExit(f"Active branch is {active!r}, expected {GIT_BRANCH!r}. "
                     "Refusing to run on the wrong branch.")
GIT_COMMIT_HASH = _run(["git", "-C", CODE_DIR, "rev-parse", "HEAD"])
GIT_COMMIT_LINE = _run(["git", "-C", CODE_DIR, "log", "-1", "--oneline"])
print(f"branch  : {active}")
print(f"HEAD    : {GIT_COMMIT_LINE}")
print(f"commit  : {GIT_COMMIT_HASH}")

# Persist the commit hash next to full-training outputs so a downloaded
# checkpoint can be traced back to its source commit later.
os.makedirs(OUTPUT_DIR_FULL, exist_ok=True)
with open(os.path.join(OUTPUT_DIR_FULL, "source_commit.txt"), "w", encoding="utf-8") as f:
    f.write(GIT_COMMIT_HASH + "\n" + GIT_COMMIT_LINE + "\n")

for p in ("fadc_3d_correct/adaptive_dilated_conv_3d.py",
          "fadc_3d_correct/ada_kernel_3d.py",
          "fadc_3d_correct/freq_select_3d.py",
          "models/unet_3d_fadc_correct.py",
          "training/train_centralized_correct.py",
          "tests/test_fadc_3d_correct.py",
          "diag_fadc_3d_correct.py"):
    assert os.path.exists(os.path.join(CODE_DIR, p)), f"missing on branch: {p}"
print("Corrected FADC3D files present on branch.")


In [ ]:
# ── 3. PULL VERIFIER + ARCHITECTURE CHECKS ─────────────────────────────
# Aborts BEFORE training if the corrected code did not land. Releases every
# temporary tensor / module before returning control to later cells.
import gc, inspect, sys, torch
sys.path.insert(0, CODE_DIR)

from fadc_3d_correct.adaptive_dilated_conv_3d import AdaptiveDilatedConv3D
from fadc_3d_correct.ada_kernel_3d import AdaKern3D
from fadc_3d_correct.freq_select_3d import FrequencySelection3D
from models.unet_3d_fadc_correct import (
    build_unet3d_fadc_correct, EXPECTED_ADAPTIVE_CONV_COUNT, MODEL_NAMES,
)

# 1) module signatures
assert AdaptiveDilatedConv3D.KERNEL_SIZE == 3
assert MODEL_NAME in MODEL_NAMES, f"MODEL_NAME {MODEL_NAME} not in {MODEL_NAMES}"

# 2) build fresh encoder-only model
m = build_unet3d_fadc_correct(MODEL_NAME, in_channels=2, out_channels=2,
                              base_filters=32, deep_supervision=DEEP_SUPERVISION).cuda()

# 3) EXACTLY 8 adaptive convs, all under enc*
n_adapt = m.count_adaptive_convs()
assert n_adapt == EXPECTED_ADAPTIVE_CONV_COUNT["encoder"] == 8, \
    f"encoder placement must yield 8 adaptive convs, got {n_adapt}"
adapt_names = m.adaptive_conv_names()
outside_enc = [n for n in adapt_names if not n.startswith("enc")]
assert not outside_enc, f"adaptive convs found outside enc*: {outside_enc}"

# 4) sanity: single base kernel per adaptive conv
for name, mod in m.named_modules():
    if isinstance(mod, AdaptiveDilatedConv3D):
        top_weights = [n for n, p in mod.named_parameters(recurse=False) if n == "weight"]
        assert top_weights == ["weight"], f"{name} has unexpected top-level kernel params: {top_weights}"

# 5) dilation list is (1, 2, 3)
sample = next(mm for mm in m.modules() if isinstance(mm, AdaptiveDilatedConv3D))
assert sample.dilation_list == (1, 2, 3), sample.dilation_list

print(f"MODEL_NAME       : {MODEL_NAME}")
print(f"adaptive convs   : {n_adapt}/8 (all under enc*)")
print(f"params           : {sum(p.numel() for p in m.parameters()):,}")
print("Pull + architecture checks passed.")

# Free the check-only model + the iterator ref before later cells run.
del sample, m
gc.collect()
torch.cuda.empty_cache()
print(f"post-check VRAM alloc: {torch.cuda.memory_allocated()/1e9:.2f} GB")


In [ ]:
# ── 4. RUN CORRECTNESS TESTS (abort on nonzero) ───────────────────────
# Invoke each test file by ABSOLUTE PATH — avoids depending on `tests/`
# being an importable package (no tests/__init__.py). Each test module
# adds the repo root to sys.path itself, so this Just Works.
import os, subprocess, sys

TEST_FILES = [
    "tests/test_fadc_3d_correct.py",       # FADC3D math correctness (35+ assertions)
    "tests/test_train_correct_utils.py",   # training loop utilities (formal-only path)
]

for rel in TEST_FILES:
    test_path = os.path.join(CODE_DIR, rel)
    assert os.path.exists(test_path), f"test file missing: {test_path}"
    print(f"---- running {rel} ----")
    res = subprocess.run(
        [sys.executable, test_path],
        cwd=CODE_DIR, capture_output=True, text=True,
    )
    print(res.stdout[-6000:])
    if res.returncode != 0:
        sys.stderr.write(res.stderr[-3000:])
        raise SystemExit(f"Test file {rel} FAILED (exit {res.returncode}) — refusing to launch training.")
    print(f"---- {rel} PASSED ----\n")
print("All correctness tests PASSED.")


In [ ]:
# ── 5. SMOKE TRAINING — 2 ep on 4 cases, small patch (abort on nonzero) ─
# No-DS variant: --deep_supervision is NEVER appended, regardless of any
# future CONFIG-cell edit. The `assert DEEP_SUPERVISION is False` at the
# bottom catches such an edit before subprocess launch.
import os, subprocess, sys

train_script = os.path.join(CODE_DIR, "training", "train_centralized_correct.py")
os.makedirs(OUTPUT_DIR_SMOKE, exist_ok=True)

cmd = [
    sys.executable, "-u", train_script,
    "--model",         MODEL_NAME,
    "--data_root",     DATA_ROOT,
    "--output_dir",    OUTPUT_DIR_SMOKE,
    "--patch_size",    str(SMOKE_PATCH_SIZE[0]), str(SMOKE_PATCH_SIZE[1]), str(SMOKE_PATCH_SIZE[2]),
    "--batch_size",    "2",
    "--warmup_epochs", "1",
    "--val_every",     "1",
    "--val_overlap",   str(VAL_OVERLAP),
    "--val_sw_batch_size", str(VAL_SW_BATCH_SIZE),
    "--seed",          str(SEED),
    "--k_att_temp_start", str(K_ATT_TEMP_START),
    "--k_att_temp_end",   str(K_ATT_TEMP_END),
    "--k_att_anneal_epochs", "1",
    "--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR,
    "--smoke_test",
]
assert DEEP_SUPERVISION is False, "This notebook is the no-DS ablation."
assert "--deep_supervision" not in cmd, \
    "Refusing to launch smoke: --deep_supervision leaked into the smoke command."
assert "--resume" not in cmd, \
    "Refusing to launch smoke: --resume leaked into the smoke command."

print("SMOKE command:\n  " + " ".join(cmd))
print("=" * 60, flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = proc.stdout.read(512)
    if not chunk: break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
proc.wait()
print(f"\nSMOKE exit code: {proc.returncode}")
if proc.returncode != 0:
    raise SystemExit(f"SMOKE training failed (exit {proc.returncode}). Refusing to launch full training.")


In [ ]:
# ── 6. SMOKE CKPT: strict=True reload + forward parity + no-DS asserts ─
# In addition to strict-loading the smoke checkpoint, this cell asserts:
#   * arch_identity["deep_supervision"] is False (checkpoint carries the
#     ablation identity we asked for);
#   * with deep_supervision=False, the model returns a single segmentation
#     tensor during training (model.train()), not a tuple/list of DS heads.
import os, torch, sys
sys.path.insert(0, CODE_DIR)
from models.unet_3d_fadc_correct import build_unet3d_fadc_correct

ckpt_path = os.path.join(OUTPUT_DIR_SMOKE, "last_checkpoint.pth")
assert os.path.exists(ckpt_path), f"smoke checkpoint missing: {ckpt_path}"
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print(f"epoch    : {ckpt.get('epoch')}")
print(f"best_dice: {ckpt.get('best_dice')}")
arch = ckpt.get("arch_identity")
print(f"arch_id  : {arch}")

# arch_identity["deep_supervision"] must be False for the no-DS ablation.
assert arch is not None, "smoke checkpoint missing arch_identity"
assert arch["deep_supervision"] is False, \
    f"smoke checkpoint arch_identity['deep_supervision'] must be False, got {arch['deep_supervision']!r}"
assert arch["model_name"] == "unet3d_fadc_encoder_correct", \
    f"smoke checkpoint model_name must be unet3d_fadc_encoder_correct, got {arch['model_name']!r}"

model = build_unet3d_fadc_correct(
    arch["model_name"],
    in_channels=arch["in_channels"], out_channels=arch["out_channels"],
    base_filters=arch["base_filters"], deep_supervision=arch["deep_supervision"],
).eval()
missing, unexpected = model.load_state_dict(ckpt["model"], strict=True)
print(f"strict load: missing={len(missing)} unexpected={len(unexpected)}")
assert not missing and not unexpected, "strict reload mismatch"

# adaptive convs still 8 after reload (defence-in-depth).
n_adapt = model.count_adaptive_convs()
assert n_adapt == 8, f"reloaded model has {n_adapt} adaptive convs, expected 8"

# Forward parity in eval mode.
x = torch.randn(1, 2, 32, 32, 16)
with torch.no_grad():
    y1 = model(x)
    y2 = model(x)
    assert not isinstance(y1, (tuple, list)), \
        f"eval() forward returned {type(y1).__name__} — expected a single tensor with DS off"
diff = (y1 - y2).abs().max().item()
assert diff < 1e-6, f"non-deterministic forward under eval(): {diff}"
print(f"forward parity  max|y1-y2| = {diff:.2e}  OK")

# Training-mode forward must ALSO return a single tensor (this is the load-
# bearing behavioural difference vs. the DS run).
model.train()
y_train = model(x)
assert not isinstance(y_train, (tuple, list)), \
    f"train() forward returned {type(y_train).__name__} — expected a single tensor with DS off"
assert torch.isfinite(y_train).all(), "train() forward produced NaN/Inf"
print(f"train() forward output type = tensor, shape = {tuple(y_train.shape)}  OK")

# Backward + optimizer step sanity — verifies the reloaded weights can be
# optimized in no-DS mode without shape/dtype errors.
opt = torch.optim.SGD(model.parameters(), lr=1e-5)
opt.zero_grad()
loss = y_train.float().pow(2).mean()
loss.backward()
opt.step()
assert torch.isfinite(loss).item(), "backward produced non-finite loss"
print(f"backward + opt.step OK  loss={loss.item():.4e}")


In [ ]:
# ── 7. GPU MEMORY PROBE — one 128x128x64 batch, no-DS variant ─────────
# If this OOMs we STOP. We do NOT silently shrink the patch or batch size.
#
# With DEEP_SUPERVISION=False the model returns a single segmentation
# tensor, so the loss sum reduces to a single term. Peak VRAM will be
# slightly lower than the DS variant because no auxiliary heads are
# instantiated. We keep the (o,) fallback so the same code exercises
# both cases without a branch.
import torch, gc, os, sys
sys.path.insert(0, CODE_DIR)
from models.unet_3d_fadc_correct import build_unet3d_fadc_correct
from torch.amp import autocast, GradScaler


def run_gpu_memory_probe() -> float:
    """Full-size forward + backward + unscaled step. Returns peak alloc (GB).

    Every tensor and module created here becomes unreachable once the function
    returns — so the caller only has to call gc.collect + empty_cache to
    reclaim VRAM. Nothing important is passed out.
    """
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()
    print(f"pre-probe VRAM alloc: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    model = build_unet3d_fadc_correct(
        MODEL_NAME, in_channels=2, out_channels=2, base_filters=32,
        deep_supervision=DEEP_SUPERVISION,
    ).cuda().train()

    scaler = GradScaler("cuda")
    x = torch.randn(BATCH_SIZE, 2, *PATCH_SIZE, device="cuda")
    with autocast("cuda"):
        y = model(x)
        # With DS off, y is a single tensor. Keep the tuple fallback so the
        # same probe works if DS is ever re-enabled here for a comparison.
        outputs = y if isinstance(y, tuple) else (y,)
        loss = sum(o.float().pow(2).mean() for o in outputs)
    scaler.scale(loss).backward()

    peak = torch.cuda.max_memory_allocated() / 1e9
    print(f"probe OK. n_outputs_in_loss={len(outputs)} "
          f"peak VRAM alloc: {peak:.2f} GB")
    return peak


try:
    _peak = run_gpu_memory_probe()
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        _peak = torch.cuda.max_memory_allocated() / 1e9
        print(f"OOM at probe. peak alloc: {_peak:.2f} GB")
        # Still clean up before raising SystemExit so the notebook state
        # remains observable.
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
        raise SystemExit(
            "Full-size probe OOMed. Not launching full training. "
            "Do NOT silently change patch_size or batch_size — the "
            "experiment contract requires the declared PATCH_SIZE/BATCH_SIZE."
        )
    raise

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"post-cleanup memory_allocated : {torch.cuda.memory_allocated()/1e9:.3f} GB")
print(f"post-cleanup memory_reserved  : {torch.cuda.memory_reserved()/1e9:.3f} GB")


In [ ]:
# ── 8. FULL TRAINING — 100 ep (formal validation only; abort on nonzero) ─
# FRESH-ONLY: no --resume is ever passed. The preflight cell has already
# guaranteed OUTPUT_DIR_FULL contains no stale checkpoint. If Kaggle times
# out mid-run, DO NOT re-launch this cell without first clearing the
# checkpoint files under OUTPUT_DIR_FULL and re-running the preflight —
# resume is not part of this ablation's contract.
import os, subprocess, sys

train_script = os.path.join(CODE_DIR, "training", "train_centralized_correct.py")
os.makedirs(OUTPUT_DIR_FULL, exist_ok=True)

cmd = [
    sys.executable, "-u", train_script,
    "--model",              MODEL_NAME,
    "--data_root",          DATA_ROOT,
    "--output_dir",         OUTPUT_DIR_FULL,
    "--epochs",             str(EPOCHS),
    "--batch_size",         str(BATCH_SIZE),
    "--num_workers",        str(NUM_WORKERS),
    "--patch_size",         str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
    "--lr",                 str(LEARNING_RATE),
    "--warmup_epochs",      str(WARMUP_EPOCHS),
    "--seed",               str(SEED),
    "--k_att_temp_start",   str(K_ATT_TEMP_START),
    "--k_att_temp_end",     str(K_ATT_TEMP_END),
    "--k_att_anneal_epochs",str(K_ATT_ANNEAL_EPOCHS),
    "--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR,
    # Formal validation only.
    "--val_every",          str(VAL_EVERY),
    "--val_overlap",        str(VAL_OVERLAP),
    "--val_sw_batch_size",  str(VAL_SW_BATCH_SIZE),
    "--checkpoint_every",   str(CHECKPOINT_EVERY),
]
# NOTE: DEEP_SUPERVISION is False for this ablation — --deep_supervision
# is a store_true flag, so omitting it is exactly how we disable DS.
# We intentionally do NOT gate on `if DEEP_SUPERVISION:` here so a future
# edit that flips the config does not silently re-enable DS. Instead we
# assert it explicitly below.
assert DEEP_SUPERVISION is False, "This notebook is the no-DS ablation."
assert "--deep_supervision" not in cmd, \
    "Refusing to launch: --deep_supervision leaked into the training command."
assert "--resume" not in cmd, \
    "Refusing to launch: --resume leaked into the training command."

# Fresh-only guard: if OUTPUT_DIR_FULL already carries any checkpoint,
# abort. Same predicate as the preflight cell (defence-in-depth in case a
# previous cell wrote a checkpoint since preflight ran).
_existing = [n for n in (os.listdir(OUTPUT_DIR_FULL) if os.path.isdir(OUTPUT_DIR_FULL) else [])
             if n.endswith(".pth")]
if _existing:
    raise SystemExit(
        "Refusing to launch: OUTPUT_DIR_FULL already contains .pth files: "
        f"{_existing}. This ablation is fresh-only."
    )

print("FRESH mode: starting from epoch 1 (no --resume passed)")
print("FULL command:\n  " + " ".join(cmd))
print("=" * 60, flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = proc.stdout.read(512)
    if not chunk: break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
proc.wait()
print(f"\nFULL exit code: {proc.returncode}")
if proc.returncode != 0:
    raise SystemExit(f"FULL training exited nonzero ({proc.returncode}). Downstream diagnostic skipped.")


In [ ]:
# ── 8c. EPOCH-N FORMAL EVAL (standalone; run before resuming training) ─
# Runs formal validation (all cases, overlap=0.5) on a specific saved
# periodic checkpoint — typically checkpoint_epoch020.pth after the first
# 20-epoch training window. Reports the checkpoint's recorded epoch and
# the git commit written next to OUTPUT_DIR_FULL. Never touches the
# checkpoint. Writes the metrics to a JSON file next to the checkpoint.
#
# Comparison target: existing overlap=0.5 formal Dice at epoch 10 = 0.5429.
# The number this cell reports IS directly comparable to that.
# It is NOT comparable to the deprecated fast/proxy Dice (e.g. 0.45).
import os, subprocess, sys

# Which periodic checkpoint to evaluate.
EP20_CHECKPOINT = os.path.join(OUTPUT_DIR_FULL, "checkpoint_epoch020.pth")

if not os.path.exists(EP20_CHECKPOINT):
    print(f"no checkpoint at {EP20_CHECKPOINT}; skipping epoch-N formal eval.")
    print("Available checkpoints in OUTPUT_DIR_FULL:")
    if os.path.isdir(OUTPUT_DIR_FULL):
        for name in sorted(os.listdir(OUTPUT_DIR_FULL)):
            if name.endswith(".pth"):
                print(f"  {name}")
else:
    # Print checkpoint identity BEFORE evaluating so the reader can trace it.
    import torch
    _ck = torch.load(EP20_CHECKPOINT, map_location="cpu", weights_only=False)
    _epoch = int(_ck.get("epoch", -1))
    _best_selfrep = float(_ck.get("best_dice", 0.0))
    _arch = _ck.get("arch_identity", {})
    del _ck
    _commit_file = os.path.join(OUTPUT_DIR_FULL, "source_commit.txt")
    _commit = "(unknown)"
    if os.path.exists(_commit_file):
        with open(_commit_file, encoding="utf-8") as fh:
            _commit = fh.read().strip().splitlines()[0] if fh else "(unknown)"
    print(f"epoch-N formal eval target")
    print(f"  checkpoint             : {EP20_CHECKPOINT}")
    print(f"  checkpoint epoch (0idx): {_epoch}   (means training epoch {_epoch+1} completed)")
    print(f"  best_dice self-reported: {_best_selfrep:.4f}")
    print(f"  git commit             : {_commit}")
    print(f"  arch_identity          : {_arch}")

    eval_script = os.path.join(CODE_DIR, "training", "evaluate_correct_checkpoint.py")
    out_json = os.path.join(OUTPUT_DIR_FULL, f"formal_eval_ep{_epoch+1:03d}.json")
    cmd = [
        sys.executable, "-u", eval_script,
        "--checkpoint",             EP20_CHECKPOINT,
        "--data_root",              DATA_ROOT,
        "--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR,
        "--patch_size",             str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
        "--overlap",                str(VAL_OVERLAP),
        "--sw_batch_size",          str(VAL_SW_BATCH_SIZE),
        "--num_workers",            str(NUM_WORKERS),
        "--out",                    out_json,
    ]
    print("EPOCH-N FORMAL EVAL command:\n  " + " ".join(cmd))
    print("=" * 60, flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    while True:
        chunk = proc.stdout.read(512)
        if not chunk: break
        sys.stdout.write(chunk.decode("utf-8", errors="replace"))
        sys.stdout.flush()
    proc.wait()
    print(f"\nEPOCH-N eval exit code: {proc.returncode}")
    print(f"metrics written to: {out_json}")


In [ ]:
# ── 9. REAL-MRI DIAGNOSTIC ON best_model.pth ──────────────────────────
import os, subprocess, sys
best_ckpt = os.path.join(OUTPUT_DIR_FULL, "best_model.pth")
val_cache = os.path.join(PREPROCESSED_CACHE_DIR, "val")
if not os.path.exists(best_ckpt):
    print(f"no best checkpoint at {best_ckpt}; skipping diagnostic.")
else:
    cmd = [
        sys.executable, os.path.join(CODE_DIR, "diag_fadc_3d_correct.py"),
        "--ckpt", best_ckpt,
        "--preprocessed_cache", val_cache,
        "--n_patches", "4",
        "--patch_size", "96", "96", "48",
    ]
    print("DIAG command:\n  " + " ".join(cmd))
    print("=" * 60, flush=True)
    subprocess.run(cmd, check=False)


In [ ]:
# ── 9b. BEST-MODEL TUMOR SEGMENTATION AND ERROR VISUALIZATION ─────────
# Read-only visualization of best_model.pth predictions on validation cases.
# Does NOT modify any checkpoint, training log, formal-eval JSON, or model
# source. Emits only visualization artifacts under VIS_OUTPUT_DIR.
#
# Uses the SAME preprocessed cache, patch size, overlap, sliding-window
# batch size, and (batch_size=1, shuffle=False) validation loader as the
# formal 306-case evaluation, so predicted masks are protocol-identical.
# The reported subset mean Dice is a visualization aid, NOT the formal metric.

import os, sys, json, re, csv, random, gc

import numpy as np
import torch
from torch.amp import autocast
from tqdm import tqdm

# ── VIS CONFIG ─────────────────────────────────────────────────────────
VIS_N_CASES          = 6           # how many patients to visualize
VIS_START_INDEX      = 0           # first val_ds index to take when deterministic
VIS_SEED             = 42          # RNG seed when VIS_USE_RANDOM_CASES=True
VIS_USE_RANDOM_CASES = False       # False = first N deterministic; True = reproducible random
                                   # (never rank by Dice — that would cherry-pick)

VIS_OUTPUT_DIR = os.path.join(OUTPUT_DIR_FULL, "best_model_visualizations")

best_ckpt = os.path.join(OUTPUT_DIR_FULL, "best_model.pth")
val_cache = os.path.join(PREPROCESSED_CACHE_DIR, "val")

if not os.path.exists(best_ckpt):
    raise SystemExit(f"No best_model.pth at {best_ckpt}; nothing to visualize.")
if not os.path.isdir(val_cache):
    raise SystemExit(f"Validation cache missing at {val_cache}.")

# Repo must be importable (cell 2 already puts CODE_DIR on sys.path; re-insert defensively).
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from models.unet_3d_fadc_correct import build_unet3d_fadc_correct
from data.mama_mia_dataset import build_centralized_loaders
from training.train_centralized_correct import make_primary_predictor
from monai.inferers import sliding_window_inference

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
from IPython.display import Image as IPyImage, display

os.makedirs(VIS_OUTPUT_DIR, exist_ok=True)

# ── LOAD CHECKPOINT (read-only) ────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device       : {device}")
print(f"checkpoint   : {best_ckpt}")

ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
arch = ckpt.get("arch_identity")
if arch is None:
    raise SystemExit("Refusing to visualize: checkpoint has no 'arch_identity'.")

ckpt_epoch_zero = int(ckpt.get("epoch", -1))              # ckpt stores 0-based completed epoch
ckpt_epoch_human = ckpt_epoch_zero + 1 if ckpt_epoch_zero >= 0 else None
ckpt_best_dice = float(ckpt.get("best_dice", 0.0))
ckpt_model_name = arch["model_name"]

print(f"epoch (human): {ckpt_epoch_human}   (ckpt.epoch={ckpt_epoch_zero})")
print(f"model_name   : {ckpt_model_name}")
print(f"best_dice (self-reported): {ckpt_best_dice:.4f}")

# Sanity: expect the corrected encoder architecture we trained.
if ckpt_model_name != "unet3d_fadc_encoder_correct":
    print(f"WARNING: expected 'unet3d_fadc_encoder_correct'; got {ckpt_model_name!r}.")

# ── REBUILD MODEL FROM CHECKPOINT METADATA (no hard-coded arch fields) ─
adakern_cfg = {
    "use_position_att": bool(arch.get("fadc_correct", {}).get("use_position_att", False))
}
model = build_unet3d_fadc_correct(
    model_name       = ckpt_model_name,
    in_channels      = int(arch["in_channels"]),
    out_channels     = int(arch["out_channels"]),
    base_filters     = int(arch["base_filters"]),
    deep_supervision = bool(arch["deep_supervision"]),
    adakern_cfg      = adakern_cfg,
).to(device)
missing, unexpected = model.load_state_dict(ckpt["model"], strict=True)
if missing or unexpected:
    raise SystemExit(f"strict load failed: missing={missing} unexpected={unexpected}")
model.eval()
predictor = make_primary_predictor(model)

# ── BUILD FORMAL-VALIDATION LOADER (same protocol as formal eval) ──────
_, val_loader = build_centralized_loaders(
    data_root              = DATA_ROOT,
    split_csv              = None,
    cache_rate             = 0.0,
    num_workers            = NUM_WORKERS,
    batch_size             = 1,
    preprocessed_cache_dir = PREPROCESSED_CACHE_DIR,
    patch_size             = tuple(PATCH_SIZE),
    seed                   = 0,
)
val_ds = val_loader.dataset
n_val  = len(val_ds)
print(f"val cases    : {n_val}")
if n_val == 0:
    raise SystemExit("Empty validation split — nothing to visualize.")

# ── SELECT CASES (deterministic; never Dice-ranked to avoid cherry-picking) ─
if VIS_USE_RANDOM_CASES:
    rng = random.Random(VIS_SEED)
    all_idx = list(range(n_val))
    rng.shuffle(all_idx)
    selected_idx = sorted(all_idx[:VIS_N_CASES])
    selection_mode = f"random (seed={VIS_SEED})"
else:
    start = max(0, min(VIS_START_INDEX, max(n_val - 1, 0)))
    end   = min(n_val, start + VIS_N_CASES)
    selected_idx = list(range(start, end))
    selection_mode = f"first {end - start} from index {start}"

# Patient IDs come from the ordered `val_ds.cases` list (built from
# sorted(glob('*.npz'))). Since the loader uses shuffle=False, index i in
# val_ds corresponds exactly to cases[i] and to iteration position i.
selected_ids = [val_ds.cases[i]["patient_id"] for i in selected_idx]
print(f"selection    : {selection_mode}")
print(f"selected ids : {selected_ids}")

# ── HELPERS ────────────────────────────────────────────────────────────
def _safe_ratio(num, den):
    return float(num) / float(den) if den > 0 else 0.0

def _binary_metrics(pred_bin, gt_bin):
    """Binary Dice / IoU / Sensitivity on class-1 tumor voxels.
    All args are int arrays containing only {0, 1}."""
    tp = int(np.logical_and(pred_bin == 1, gt_bin == 1).sum())
    fp = int(np.logical_and(pred_bin == 1, gt_bin == 0).sum())
    fn = int(np.logical_and(pred_bin == 0, gt_bin == 1).sum())
    gt_vol   = int((gt_bin == 1).sum())
    pred_vol = int((pred_bin == 1).sum())
    return {
        "dice":              _safe_ratio(2 * tp, 2 * tp + fp + fn),
        "iou":               _safe_ratio(tp, tp + fp + fn),
        "sensitivity":       _safe_ratio(tp, tp + fn),
        "tp_voxels":         tp,
        "fp_voxels":         fp,
        "fn_voxels":         fn,
        "gt_tumor_voxels":   gt_vol,
        "pred_tumor_voxels": pred_vol,
    }

def _pick_slice(gt_bin, pred_bin):
    """Return (slice_index, reason) along the LAST spatial axis (axial z).
    Preprocessed cache stores volumes as (C, H, W, D) so gt_bin.shape[-1] = D."""
    depth = int(gt_bin.shape[-1])
    if (gt_bin == 1).any():
        areas = (gt_bin == 1).sum(axis=(0, 1))
        return int(np.argmax(areas)), "max_gt_area"
    if (pred_bin == 1).any():
        areas = (pred_bin == 1).sum(axis=(0, 1))
        return int(np.argmax(areas)), "max_pred_area"
    return depth // 2, "center_slice"

def _robust_norm(img2d):
    """Percentile-based intensity normalization for DISPLAY ONLY.
    Does not touch the model input."""
    lo, hi = np.percentile(img2d, (1.0, 99.0))
    if hi <= lo:
        lo, hi = float(img2d.min()), float(img2d.max())
        if hi <= lo:
            return np.zeros_like(img2d, dtype=np.float32)
    return np.clip((img2d - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)

def _sanitize(pid):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(pid))

# Colour maps
_gt_cmap   = ListedColormap([(0, 0, 0, 0), (0.10, 0.85, 0.10, 0.55)])   # transparent → green
_pred_cmap = ListedColormap([(0, 0, 0, 0), (0.10, 0.85, 0.90, 0.55)])   # transparent → cyan
_err_cmap  = ListedColormap([
    (0, 0, 0, 0),                 # 0 background: transparent
    (1.00, 0.92, 0.10, 0.75),     # 1 TP: yellow
    (0.95, 0.15, 0.15, 0.75),     # 2 FP: red
    (0.20, 1.00, 0.20, 0.75),     # 3 FN: lime green
])
_err_legend = [
    Patch(facecolor=(1.00, 0.92, 0.10), edgecolor="black", label="True positive"),
    Patch(facecolor=(0.95, 0.15, 0.15), edgecolor="black", label="False positive"),
    Patch(facecolor=(0.20, 1.00, 0.20), edgecolor="black", label="False negative"),
]

col_titles = [
    "Post-contrast MRI",
    "MRI + GT (green)",
    "MRI + Prediction (cyan)",
    "MRI + Error map",
    "GT vs Pred (contours)",
]

# ── RUN INFERENCE ON THE SELECTED CASES ────────────────────────────────
per_patient = []
device_type = "cuda" if device.type == "cuda" else "cpu"
use_amp     = device.type == "cuda"

# Fetch cases directly from the dataset (is_train=False → no augmentation,
# no random crop). This lets us grab specific indices without iterating
# the whole 306-case loader for a 6-case visualization.
with torch.no_grad():
    pbar = tqdm(selected_idx, desc="vis inference", unit="case",
                file=sys.stdout, dynamic_ncols=False, ncols=100)
    for idx in pbar:
        sample  = val_ds[idx]
        image_t = sample["image"]
        label_t = sample["label"]
        if not torch.is_tensor(image_t):
            image_t = torch.as_tensor(np.asarray(image_t))
        if not torch.is_tensor(label_t):
            label_t = torch.as_tensor(np.asarray(label_t))
        image_t = image_t.float()
        label_t = label_t.long()

        # Preprocessed cache stores (C=2, H, W, D). Sanity-check.
        if image_t.ndim != 4 or image_t.shape[0] != 2:
            raise SystemExit(
                f"Unexpected image shape {tuple(image_t.shape)} "
                f"for {val_ds.cases[idx]['patient_id']} — expected (2, H, W, D)."
            )
        # Label may be (1, H, W, D) or (H, W, D). Normalize to (H, W, D) class indices.
        if label_t.ndim == 4:
            label_np = label_t[0].cpu().numpy().astype(np.int64)
        elif label_t.ndim == 3:
            label_np = label_t.cpu().numpy().astype(np.int64)
        else:
            raise SystemExit(f"unexpected label ndim={label_t.ndim}")

        # Sliding-window inference (autocast on CUDA), argmax → class ids. No threshold tuning.
        inputs = image_t.unsqueeze(0).to(device, non_blocking=True)  # (1, 2, H, W, D)
        with autocast(device_type, enabled=use_amp):
            logits = sliding_window_inference(
                inputs        = inputs,
                roi_size      = tuple(PATCH_SIZE),
                sw_batch_size = VAL_SW_BATCH_SIZE,
                predictor     = predictor,
                overlap       = VAL_OVERLAP,
            )
        # Defensive: if any deep-supervision tuple slipped through in eval mode.
        if isinstance(logits, (tuple, list)):
            logits = logits[0]

        pred_ids = torch.argmax(logits[0], dim=0).cpu().numpy().astype(np.int64)  # (H, W, D)
        pred_bin = (pred_ids == 1).astype(np.int64)
        gt_bin   = (label_np == 1).astype(np.int64)

        # Post-contrast MRI channel for display (image[1]).
        post_np = image_t[1].cpu().numpy()

        metrics = _binary_metrics(pred_bin, gt_bin)
        z, z_reason = _pick_slice(gt_bin, pred_bin)
        rec = {
            "patient_id":   val_ds.cases[idx]["patient_id"],
            "val_index":    int(idx),
            "slice_index":  int(z),
            "slice_reason": z_reason,
            **metrics,
            "_post_slice":  post_np[:, :, z].astype(np.float32),
            "_gt_slice":    gt_bin[:, :, z].astype(np.uint8),
            "_pred_slice":  pred_bin[:, :, z].astype(np.uint8),
        }
        per_patient.append(rec)
        pbar.set_postfix({"dice": f"{metrics['dice']:.3f}"})

        # Release per-case GPU tensors before the next iteration.
        del inputs, logits
        if device.type == "cuda":
            torch.cuda.empty_cache()
    pbar.close()

gc.collect()

# ── MONTAGE FIGURE (rows = patients, cols = 5) ─────────────────────────
n_rows = len(per_patient)
fig, axes = plt.subplots(n_rows, 5, figsize=(22.0, 4.4 * max(n_rows, 1)),
                         squeeze=False)

for r, rec in enumerate(per_patient):
    bg   = _robust_norm(rec["_post_slice"])
    gt2d = rec["_gt_slice"]
    pd2d = rec["_pred_slice"]

    err_map = np.zeros_like(gt2d, dtype=np.uint8)
    err_map[(pd2d == 1) & (gt2d == 1)] = 1   # TP
    err_map[(pd2d == 1) & (gt2d == 0)] = 2   # FP
    err_map[(pd2d == 0) & (gt2d == 1)] = 3   # FN

    for c in range(5):
        axes[r][c].imshow(bg, cmap="gray", vmin=0.0, vmax=1.0)
        axes[r][c].set_xticks([]); axes[r][c].set_yticks([])
        for spine in axes[r][c].spines.values():
            spine.set_visible(False)

    axes[r][1].imshow(gt2d,    cmap=_gt_cmap,   vmin=0, vmax=1, interpolation="nearest")
    axes[r][2].imshow(pd2d,    cmap=_pred_cmap, vmin=0, vmax=1, interpolation="nearest")
    axes[r][3].imshow(err_map, cmap=_err_cmap,  vmin=0, vmax=3, interpolation="nearest")
    if gt2d.any():
        axes[r][4].contour(gt2d, levels=[0.5], colors=["#1BE01B"], linewidths=1.4)
    if pd2d.any():
        axes[r][4].contour(pd2d, levels=[0.5], colors=["#1BD8E8"], linewidths=1.4, linestyles="--")

    row_label = (
        f"{rec['patient_id']}\n"
        f"z={rec['slice_index']} ({rec['slice_reason']})\n"
        f"Dice={rec['dice']:.3f}  IoU={rec['iou']:.3f}  Sens={rec['sensitivity']:.3f}\n"
        f"TP={rec['tp_voxels']}  FP={rec['fp_voxels']}  FN={rec['fn_voxels']}"
    )
    axes[r][0].set_ylabel(row_label, rotation=0, ha="right", va="center",
                          fontsize=9, labelpad=115)

    if r == 0:
        for c, t in enumerate(col_titles):
            axes[r][c].set_title(t, fontsize=11)

axes[-1][3].legend(handles=_err_legend, loc="lower center",
                   bbox_to_anchor=(0.5, -0.22), ncol=3, fontsize=9, frameon=False)

fig.suptitle(
    f"best_model.pth  |  ep {ckpt_epoch_human}  |  "
    f"self-reported best_dice = {ckpt_best_dice:.4f}\n"
    f"visualization subset (n={n_rows}) — NOT the formal 306-case metric",
    fontsize=13, y=0.995,
)
plt.tight_layout(rect=[0.065, 0.02, 1.0, 0.97])

montage_path = os.path.join(VIS_OUTPUT_DIR, "best_model_error_montage.png")
fig.savefig(montage_path, dpi=200, bbox_inches="tight")
plt.close(fig)

# ── ONE PNG PER PATIENT ────────────────────────────────────────────────
individual_paths = []
for rec in per_patient:
    bg   = _robust_norm(rec["_post_slice"])
    gt2d = rec["_gt_slice"]
    pd2d = rec["_pred_slice"]
    err_map = np.zeros_like(gt2d, dtype=np.uint8)
    err_map[(pd2d == 1) & (gt2d == 1)] = 1
    err_map[(pd2d == 1) & (gt2d == 0)] = 2
    err_map[(pd2d == 0) & (gt2d == 1)] = 3

    ifig, iax = plt.subplots(1, 5, figsize=(22.0, 4.8))
    for c in range(5):
        iax[c].imshow(bg, cmap="gray", vmin=0.0, vmax=1.0)
        iax[c].axis("off")
    iax[1].imshow(gt2d,    cmap=_gt_cmap,   vmin=0, vmax=1, interpolation="nearest")
    iax[2].imshow(pd2d,    cmap=_pred_cmap, vmin=0, vmax=1, interpolation="nearest")
    iax[3].imshow(err_map, cmap=_err_cmap,  vmin=0, vmax=3, interpolation="nearest")
    if gt2d.any():
        iax[4].contour(gt2d, levels=[0.5], colors=["#1BE01B"], linewidths=1.4)
    if pd2d.any():
        iax[4].contour(pd2d, levels=[0.5], colors=["#1BD8E8"], linewidths=1.4, linestyles="--")
    for c, t in enumerate(col_titles):
        iax[c].set_title(t, fontsize=11)
    iax[3].legend(handles=_err_legend, loc="lower center",
                  bbox_to_anchor=(0.5, -0.20), ncol=3, fontsize=9, frameon=False)
    ifig.suptitle(
        f"{rec['patient_id']}  |  z={rec['slice_index']} ({rec['slice_reason']})  |  "
        f"Dice={rec['dice']:.3f}  IoU={rec['iou']:.3f}  Sens={rec['sensitivity']:.3f}  |  "
        f"TP={rec['tp_voxels']}  FP={rec['fp_voxels']}  FN={rec['fn_voxels']}",
        fontsize=11, y=1.02,
    )
    plt.tight_layout()
    ipath = os.path.join(VIS_OUTPUT_DIR, f"patient_{_sanitize(rec['patient_id'])}.png")
    ifig.savefig(ipath, dpi=200, bbox_inches="tight")
    plt.close(ifig)
    individual_paths.append(ipath)

# ── METRICS PERSISTENCE (CSV + JSON) ───────────────────────────────────
csv_path  = os.path.join(VIS_OUTPUT_DIR, "best_model_visualization_metrics.csv")
json_path = os.path.join(VIS_OUTPUT_DIR, "best_model_visualization_metrics.json")

_csv_fields = ["patient_id", "val_index", "slice_index", "slice_reason",
               "dice", "iou", "sensitivity",
               "tp_voxels", "fp_voxels", "fn_voxels",
               "gt_tumor_voxels", "pred_tumor_voxels"]
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=_csv_fields)
    w.writeheader()
    for rec in per_patient:
        w.writerow({k: rec[k] for k in _csv_fields})

subset_dice_mean = float(np.mean([r["dice"] for r in per_patient])) if per_patient else 0.0
json_payload = {
    "checkpoint":                        os.path.abspath(best_ckpt),
    "checkpoint_epoch_human":            ckpt_epoch_human,
    "checkpoint_epoch_zero_based":       ckpt_epoch_zero,
    "checkpoint_best_dice_selfreported": ckpt_best_dice,
    "model_name":                        ckpt_model_name,
    "arch_identity":                     arch,
    "vis_config": {
        "VIS_N_CASES":          VIS_N_CASES,
        "VIS_START_INDEX":      VIS_START_INDEX,
        "VIS_SEED":             VIS_SEED,
        "VIS_USE_RANDOM_CASES": VIS_USE_RANDOM_CASES,
        "selection_mode":       selection_mode,
    },
    "patch_size":                    list(PATCH_SIZE),
    "overlap":                       VAL_OVERLAP,
    "sw_batch_size":                 VAL_SW_BATCH_SIZE,
    "n_visualized":                  len(per_patient),
    "visualized_ids":                [r["patient_id"] for r in per_patient],
    "subset_mean_dice_NOT_FORMAL":   subset_dice_mean,
    "per_patient":                   [{k: r[k] for k in _csv_fields} for r in per_patient],
    "outputs": {
        "montage":         montage_path,
        "csv":             csv_path,
        "json":            json_path,
        "per_patient_png": individual_paths,
    },
}
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_payload, f, indent=2)

# ── FINAL REPORT (stdout) ──────────────────────────────────────────────
print()
print("=" * 72)
print("VISUALIZATION SUMMARY")
print("=" * 72)
print(f"checkpoint                : {best_ckpt}")
print(f"epoch (human-readable)    : {ckpt_epoch_human}")
print(f"best_dice (self-reported) : {ckpt_best_dice:.4f}")
print(f"selection                 : {selection_mode}")
print(f"visualized patient ids    : {[r['patient_id'] for r in per_patient]}")
print()
print("per-patient metrics:")
for r in per_patient:
    print(f"  {r['patient_id']:24s}  z={r['slice_index']:3d}  "
          f"Dice={r['dice']:.3f}  IoU={r['iou']:.3f}  Sens={r['sensitivity']:.3f}  "
          f"TP={r['tp_voxels']:>7d}  FP={r['fp_voxels']:>7d}  FN={r['fn_voxels']:>7d}")
print()
print(f"subset mean Dice          : {subset_dice_mean:.4f}   "
      f"(n={len(per_patient)} — VISUALIZATION SUBSET, NOT the formal 306-case metric)")
print()
print("output files:")
print(f"  montage : {montage_path}")
print(f"  csv     : {csv_path}")
print(f"  json    : {json_path}")
for p in individual_paths:
    print(f"  case    : {p}")

# Display the montage inline in the notebook.
display(IPyImage(filename=montage_path))


In [ ]:
# ── 9c. BEST-MODEL PER-COLLECTION VISUALIZATION ───────────────────────
# Same read-only discipline as cell 9b, but selects ONE case per collection
# (DUKE / ISPY1 / ISPY2 / NACT) instead of N consecutive cases from a single
# collection. Useful for showing generalization across the four MAMA-MIA
# collections at a glance.
#
# Emits its own artifacts under a separate subfolder so 9b outputs are not
# clobbered. Independent of 9b — reload/reuse safe regardless of run order.

import os, sys, json, re, csv, random, gc
from collections import OrderedDict

import numpy as np
import torch
from torch.amp import autocast
from tqdm import tqdm

# ── VIS CONFIG ─────────────────────────────────────────────────────────
VIS_PC_PER_COLLECTION      = 1       # cases to visualize per collection
VIS_PC_SEED                = 42      # seed used when random selection is on
VIS_PC_USE_RANDOM_PER_COLL = False   # False = first N per collection in dataset order;
                                     # True  = reproducibly-random within each collection.
                                     # (Never rank by Dice — that would cherry-pick.)
VIS_PC_COLLECTIONS         = ["DUKE", "ISPY1", "ISPY2", "NACT"]

VIS_PC_OUTPUT_DIR = os.path.join(OUTPUT_DIR_FULL, "best_model_visualizations_per_collection")

best_ckpt = os.path.join(OUTPUT_DIR_FULL, "best_model.pth")
val_cache = os.path.join(PREPROCESSED_CACHE_DIR, "val")
if not os.path.exists(best_ckpt):
    raise SystemExit(f"No best_model.pth at {best_ckpt}; nothing to visualize.")
if not os.path.isdir(val_cache):
    raise SystemExit(f"Validation cache missing at {val_cache}.")

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from models.unet_3d_fadc_correct import build_unet3d_fadc_correct
from data.mama_mia_dataset import build_centralized_loaders
from training.train_centralized_correct import make_primary_predictor
from monai.inferers import sliding_window_inference

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
from IPython.display import Image as IPyImage, display

os.makedirs(VIS_PC_OUTPUT_DIR, exist_ok=True)

# ── LOAD CHECKPOINT (read-only) ────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device       : {device}")
print(f"checkpoint   : {best_ckpt}")

ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
arch = ckpt.get("arch_identity")
if arch is None:
    raise SystemExit("Refusing to visualize: checkpoint has no 'arch_identity'.")

ckpt_epoch_zero  = int(ckpt.get("epoch", -1))
ckpt_epoch_human = ckpt_epoch_zero + 1 if ckpt_epoch_zero >= 0 else None
ckpt_best_dice   = float(ckpt.get("best_dice", 0.0))
ckpt_model_name  = arch["model_name"]

print(f"epoch (human): {ckpt_epoch_human}   (ckpt.epoch={ckpt_epoch_zero})")
print(f"model_name   : {ckpt_model_name}")
print(f"best_dice (self-reported): {ckpt_best_dice:.4f}")

if ckpt_model_name != "unet3d_fadc_encoder_correct":
    print(f"WARNING: expected 'unet3d_fadc_encoder_correct'; got {ckpt_model_name!r}.")

# ── REBUILD MODEL FROM CHECKPOINT METADATA (no hard-coded arch fields) ─
adakern_cfg = {
    "use_position_att": bool(arch.get("fadc_correct", {}).get("use_position_att", False))
}
model = build_unet3d_fadc_correct(
    model_name       = ckpt_model_name,
    in_channels      = int(arch["in_channels"]),
    out_channels     = int(arch["out_channels"]),
    base_filters     = int(arch["base_filters"]),
    deep_supervision = bool(arch["deep_supervision"]),
    adakern_cfg      = adakern_cfg,
).to(device)
missing, unexpected = model.load_state_dict(ckpt["model"], strict=True)
if missing or unexpected:
    raise SystemExit(f"strict load failed: missing={missing} unexpected={unexpected}")
model.eval()
predictor = make_primary_predictor(model)

# ── BUILD VAL LOADER (same protocol as formal eval) ────────────────────
_, val_loader = build_centralized_loaders(
    data_root              = DATA_ROOT,
    split_csv              = None,
    cache_rate             = 0.0,
    num_workers            = NUM_WORKERS,
    batch_size             = 1,
    preprocessed_cache_dir = PREPROCESSED_CACHE_DIR,
    patch_size             = tuple(PATCH_SIZE),
    seed                   = 0,
)
val_ds = val_loader.dataset
n_val  = len(val_ds)
print(f"val cases    : {n_val}")
if n_val == 0:
    raise SystemExit("Empty validation split — nothing to visualize.")

# ── GROUP CASES BY COLLECTION (order = dataset order) ──────────────────
# val_ds.cases order is stable — driven by sorted(glob('*.npz')) in
# discover_cases_from_cache — so per-collection groups are deterministic.
by_collection: "OrderedDict[str, list[int]]" = OrderedDict((c, []) for c in VIS_PC_COLLECTIONS)
extras_by_collection: "OrderedDict[str, list[int]]" = OrderedDict()
for idx, c in enumerate(val_ds.cases):
    coll = c.get("collection")
    if coll in by_collection:
        by_collection[coll].append(idx)
    else:
        extras_by_collection.setdefault(coll, []).append(idx)

print()
print("val cases by collection:")
for coll, idxs in by_collection.items():
    print(f"  {coll:8s} : {len(idxs):4d}")
for coll, idxs in extras_by_collection.items():
    print(f"  {coll!s:8s} : {len(idxs):4d}   (unexpected — not in VIS_PC_COLLECTIONS)")

selected_idx: list[int] = []
selection_notes: list[str] = []
if VIS_PC_USE_RANDOM_PER_COLL:
    rng = random.Random(VIS_PC_SEED)
    for coll, idxs in by_collection.items():
        if not idxs:
            selection_notes.append(f"{coll}: NO CASES in val — skipping")
            continue
        pool = list(idxs)
        rng.shuffle(pool)
        pick = pool[:VIS_PC_PER_COLLECTION]
        selected_idx.extend(pick)
        selection_notes.append(f"{coll}: random {pick} (seed={VIS_PC_SEED})")
    selection_mode = f"random-per-collection (seed={VIS_PC_SEED}, k={VIS_PC_PER_COLLECTION})"
else:
    for coll, idxs in by_collection.items():
        if not idxs:
            selection_notes.append(f"{coll}: NO CASES in val — skipping")
            continue
        pick = idxs[:VIS_PC_PER_COLLECTION]
        selected_idx.extend(pick)
        selection_notes.append(f"{coll}: first {pick}")
    selection_mode = f"first-{VIS_PC_PER_COLLECTION}-per-collection (dataset order)"

if not selected_idx:
    raise SystemExit("No cases selected — every configured collection is empty.")

selected_ids = [val_ds.cases[i]["patient_id"] for i in selected_idx]
selected_colls = [val_ds.cases[i].get("collection", "?") for i in selected_idx]
print()
print(f"selection    : {selection_mode}")
for note in selection_notes:
    print(f"  {note}")
print(f"selected ids : {list(zip(selected_colls, selected_ids))}")

# ── HELPERS (self-contained; do NOT depend on cell 9b having run) ──────
def _safe_ratio_pc(num, den):
    return float(num) / float(den) if den > 0 else 0.0

def _binary_metrics_pc(pred_bin, gt_bin):
    tp = int(np.logical_and(pred_bin == 1, gt_bin == 1).sum())
    fp = int(np.logical_and(pred_bin == 1, gt_bin == 0).sum())
    fn = int(np.logical_and(pred_bin == 0, gt_bin == 1).sum())
    return {
        "dice":              _safe_ratio_pc(2 * tp, 2 * tp + fp + fn),
        "iou":               _safe_ratio_pc(tp, tp + fp + fn),
        "sensitivity":       _safe_ratio_pc(tp, tp + fn),
        "tp_voxels":         tp,
        "fp_voxels":         fp,
        "fn_voxels":         fn,
        "gt_tumor_voxels":   int((gt_bin == 1).sum()),
        "pred_tumor_voxels": int((pred_bin == 1).sum()),
    }

def _pick_slice_pc(gt_bin, pred_bin):
    depth = int(gt_bin.shape[-1])
    if (gt_bin == 1).any():
        return int(np.argmax((gt_bin == 1).sum(axis=(0, 1)))), "max_gt_area"
    if (pred_bin == 1).any():
        return int(np.argmax((pred_bin == 1).sum(axis=(0, 1)))), "max_pred_area"
    return depth // 2, "center_slice"

def _robust_norm_pc(img2d):
    lo, hi = np.percentile(img2d, (1.0, 99.0))
    if hi <= lo:
        lo, hi = float(img2d.min()), float(img2d.max())
        if hi <= lo:
            return np.zeros_like(img2d, dtype=np.float32)
    return np.clip((img2d - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)

def _sanitize_pc(pid):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(pid))

_gt_cmap_pc   = ListedColormap([(0, 0, 0, 0), (0.10, 0.85, 0.10, 0.55)])
_pred_cmap_pc = ListedColormap([(0, 0, 0, 0), (0.10, 0.85, 0.90, 0.55)])
_err_cmap_pc  = ListedColormap([
    (0, 0, 0, 0),
    (1.00, 0.92, 0.10, 0.75),   # TP  yellow
    (0.95, 0.15, 0.15, 0.75),   # FP  red
    (0.20, 1.00, 0.20, 0.75),   # FN  lime
])
_err_legend_pc = [
    Patch(facecolor=(1.00, 0.92, 0.10), edgecolor="black", label="True positive"),
    Patch(facecolor=(0.95, 0.15, 0.15), edgecolor="black", label="False positive"),
    Patch(facecolor=(0.20, 1.00, 0.20), edgecolor="black", label="False negative"),
]
col_titles_pc = [
    "Post-contrast MRI",
    "MRI + GT (green)",
    "MRI + Prediction (cyan)",
    "MRI + Error map",
    "GT vs Pred (contours)",
]

# ── INFERENCE ──────────────────────────────────────────────────────────
per_patient = []
device_type = "cuda" if device.type == "cuda" else "cpu"
use_amp     = device.type == "cuda"

with torch.no_grad():
    pbar = tqdm(selected_idx, desc="vis-per-coll inference", unit="case",
                file=sys.stdout, dynamic_ncols=False, ncols=100)
    for idx in pbar:
        sample  = val_ds[idx]
        image_t = sample["image"]
        label_t = sample["label"]
        if not torch.is_tensor(image_t):
            image_t = torch.as_tensor(np.asarray(image_t))
        if not torch.is_tensor(label_t):
            label_t = torch.as_tensor(np.asarray(label_t))
        image_t = image_t.float()
        label_t = label_t.long()

        if image_t.ndim != 4 or image_t.shape[0] != 2:
            raise SystemExit(
                f"Unexpected image shape {tuple(image_t.shape)} "
                f"for {val_ds.cases[idx]['patient_id']} — expected (2, H, W, D)."
            )
        if label_t.ndim == 4:
            label_np = label_t[0].cpu().numpy().astype(np.int64)
        elif label_t.ndim == 3:
            label_np = label_t.cpu().numpy().astype(np.int64)
        else:
            raise SystemExit(f"unexpected label ndim={label_t.ndim}")

        inputs = image_t.unsqueeze(0).to(device, non_blocking=True)
        with autocast(device_type, enabled=use_amp):
            logits = sliding_window_inference(
                inputs        = inputs,
                roi_size      = tuple(PATCH_SIZE),
                sw_batch_size = VAL_SW_BATCH_SIZE,
                predictor     = predictor,
                overlap       = VAL_OVERLAP,
            )
        if isinstance(logits, (tuple, list)):
            logits = logits[0]

        pred_ids = torch.argmax(logits[0], dim=0).cpu().numpy().astype(np.int64)
        pred_bin = (pred_ids == 1).astype(np.int64)
        gt_bin   = (label_np == 1).astype(np.int64)
        post_np  = image_t[1].cpu().numpy()

        metrics = _binary_metrics_pc(pred_bin, gt_bin)
        z, z_reason = _pick_slice_pc(gt_bin, pred_bin)
        per_patient.append({
            "collection":   val_ds.cases[idx].get("collection", "?"),
            "patient_id":   val_ds.cases[idx]["patient_id"],
            "val_index":    int(idx),
            "slice_index":  int(z),
            "slice_reason": z_reason,
            **metrics,
            "_post_slice":  post_np[:, :, z].astype(np.float32),
            "_gt_slice":    gt_bin[:, :, z].astype(np.uint8),
            "_pred_slice":  pred_bin[:, :, z].astype(np.uint8),
        })
        pbar.set_postfix({"coll": val_ds.cases[idx].get("collection", "?"),
                          "dice": f"{metrics['dice']:.3f}"})

        del inputs, logits
        if device.type == "cuda":
            torch.cuda.empty_cache()
    pbar.close()

gc.collect()

# Group rows by the order of VIS_PC_COLLECTIONS so the montage reads DUKE→NACT.
def _coll_order_key(rec):
    try:
        return (VIS_PC_COLLECTIONS.index(rec["collection"]), rec["val_index"])
    except ValueError:
        return (len(VIS_PC_COLLECTIONS), rec["val_index"])

per_patient.sort(key=_coll_order_key)

# ── MONTAGE FIGURE ─────────────────────────────────────────────────────
n_rows = len(per_patient)
fig, axes = plt.subplots(n_rows, 5, figsize=(22.0, 4.4 * max(n_rows, 1)),
                         squeeze=False)

for r, rec in enumerate(per_patient):
    bg   = _robust_norm_pc(rec["_post_slice"])
    gt2d = rec["_gt_slice"]
    pd2d = rec["_pred_slice"]

    err_map = np.zeros_like(gt2d, dtype=np.uint8)
    err_map[(pd2d == 1) & (gt2d == 1)] = 1   # TP
    err_map[(pd2d == 1) & (gt2d == 0)] = 2   # FP
    err_map[(pd2d == 0) & (gt2d == 1)] = 3   # FN

    for c in range(5):
        axes[r][c].imshow(bg, cmap="gray", vmin=0.0, vmax=1.0)
        axes[r][c].set_xticks([]); axes[r][c].set_yticks([])
        for spine in axes[r][c].spines.values():
            spine.set_visible(False)

    axes[r][1].imshow(gt2d,    cmap=_gt_cmap_pc,   vmin=0, vmax=1, interpolation="nearest")
    axes[r][2].imshow(pd2d,    cmap=_pred_cmap_pc, vmin=0, vmax=1, interpolation="nearest")
    axes[r][3].imshow(err_map, cmap=_err_cmap_pc,  vmin=0, vmax=3, interpolation="nearest")
    if gt2d.any():
        axes[r][4].contour(gt2d, levels=[0.5], colors=["#1BE01B"], linewidths=1.4)
    if pd2d.any():
        axes[r][4].contour(pd2d, levels=[0.5], colors=["#1BD8E8"], linewidths=1.4, linestyles="--")

    row_label = (
        f"[{rec['collection']}]\n"
        f"{rec['patient_id']}\n"
        f"z={rec['slice_index']} ({rec['slice_reason']})\n"
        f"Dice={rec['dice']:.3f}  IoU={rec['iou']:.3f}  Sens={rec['sensitivity']:.3f}\n"
        f"TP={rec['tp_voxels']}  FP={rec['fp_voxels']}  FN={rec['fn_voxels']}"
    )
    axes[r][0].set_ylabel(row_label, rotation=0, ha="right", va="center",
                          fontsize=9, labelpad=115)

    if r == 0:
        for c, t in enumerate(col_titles_pc):
            axes[r][c].set_title(t, fontsize=11)

axes[-1][3].legend(handles=_err_legend_pc, loc="lower center",
                   bbox_to_anchor=(0.5, -0.22), ncol=3, fontsize=9, frameon=False)

fig.suptitle(
    f"best_model.pth  |  ep {ckpt_epoch_human}  |  "
    f"self-reported best_dice = {ckpt_best_dice:.4f}\n"
    f"one-per-collection visualization (n={n_rows}) — NOT the formal 306-case metric",
    fontsize=13, y=0.995,
)
plt.tight_layout(rect=[0.065, 0.02, 1.0, 0.97])

montage_path = os.path.join(VIS_PC_OUTPUT_DIR, "best_model_error_montage_per_collection.png")
fig.savefig(montage_path, dpi=200, bbox_inches="tight")
plt.close(fig)

# ── ONE PNG PER PATIENT (filename carries collection prefix) ───────────
individual_paths = []
for rec in per_patient:
    bg   = _robust_norm_pc(rec["_post_slice"])
    gt2d = rec["_gt_slice"]
    pd2d = rec["_pred_slice"]
    err_map = np.zeros_like(gt2d, dtype=np.uint8)
    err_map[(pd2d == 1) & (gt2d == 1)] = 1
    err_map[(pd2d == 1) & (gt2d == 0)] = 2
    err_map[(pd2d == 0) & (gt2d == 1)] = 3

    ifig, iax = plt.subplots(1, 5, figsize=(22.0, 4.8))
    for c in range(5):
        iax[c].imshow(bg, cmap="gray", vmin=0.0, vmax=1.0)
        iax[c].axis("off")
    iax[1].imshow(gt2d,    cmap=_gt_cmap_pc,   vmin=0, vmax=1, interpolation="nearest")
    iax[2].imshow(pd2d,    cmap=_pred_cmap_pc, vmin=0, vmax=1, interpolation="nearest")
    iax[3].imshow(err_map, cmap=_err_cmap_pc,  vmin=0, vmax=3, interpolation="nearest")
    if gt2d.any():
        iax[4].contour(gt2d, levels=[0.5], colors=["#1BE01B"], linewidths=1.4)
    if pd2d.any():
        iax[4].contour(pd2d, levels=[0.5], colors=["#1BD8E8"], linewidths=1.4, linestyles="--")
    for c, t in enumerate(col_titles_pc):
        iax[c].set_title(t, fontsize=11)
    iax[3].legend(handles=_err_legend_pc, loc="lower center",
                  bbox_to_anchor=(0.5, -0.20), ncol=3, fontsize=9, frameon=False)
    ifig.suptitle(
        f"[{rec['collection']}] {rec['patient_id']}  |  "
        f"z={rec['slice_index']} ({rec['slice_reason']})  |  "
        f"Dice={rec['dice']:.3f}  IoU={rec['iou']:.3f}  Sens={rec['sensitivity']:.3f}  |  "
        f"TP={rec['tp_voxels']}  FP={rec['fp_voxels']}  FN={rec['fn_voxels']}",
        fontsize=11, y=1.02,
    )
    plt.tight_layout()
    fname = f"collection_{_sanitize_pc(rec['collection'])}__patient_{_sanitize_pc(rec['patient_id'])}.png"
    ipath = os.path.join(VIS_PC_OUTPUT_DIR, fname)
    ifig.savefig(ipath, dpi=200, bbox_inches="tight")
    plt.close(ifig)
    individual_paths.append(ipath)

# ── METRICS PERSISTENCE (CSV + JSON) ───────────────────────────────────
csv_path  = os.path.join(VIS_PC_OUTPUT_DIR, "best_model_visualization_per_collection_metrics.csv")
json_path = os.path.join(VIS_PC_OUTPUT_DIR, "best_model_visualization_per_collection_metrics.json")

_csv_fields = ["collection", "patient_id", "val_index", "slice_index", "slice_reason",
               "dice", "iou", "sensitivity",
               "tp_voxels", "fp_voxels", "fn_voxels",
               "gt_tumor_voxels", "pred_tumor_voxels"]
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=_csv_fields)
    w.writeheader()
    for rec in per_patient:
        w.writerow({k: rec[k] for k in _csv_fields})

subset_dice_mean = float(np.mean([r["dice"] for r in per_patient])) if per_patient else 0.0
per_collection_dice = {}
for rec in per_patient:
    per_collection_dice.setdefault(rec["collection"], []).append(rec["dice"])
per_collection_dice_mean = {c: float(np.mean(v)) for c, v in per_collection_dice.items()}

json_payload = {
    "checkpoint":                        os.path.abspath(best_ckpt),
    "checkpoint_epoch_human":            ckpt_epoch_human,
    "checkpoint_epoch_zero_based":       ckpt_epoch_zero,
    "checkpoint_best_dice_selfreported": ckpt_best_dice,
    "model_name":                        ckpt_model_name,
    "arch_identity":                     arch,
    "vis_config": {
        "VIS_PC_PER_COLLECTION":      VIS_PC_PER_COLLECTION,
        "VIS_PC_SEED":                VIS_PC_SEED,
        "VIS_PC_USE_RANDOM_PER_COLL": VIS_PC_USE_RANDOM_PER_COLL,
        "VIS_PC_COLLECTIONS":         VIS_PC_COLLECTIONS,
        "selection_mode":             selection_mode,
        "selection_notes":            selection_notes,
    },
    "patch_size":                          list(PATCH_SIZE),
    "overlap":                             VAL_OVERLAP,
    "sw_batch_size":                       VAL_SW_BATCH_SIZE,
    "n_visualized":                        len(per_patient),
    "visualized":                          [{"collection": r["collection"],
                                             "patient_id": r["patient_id"]}
                                            for r in per_patient],
    "subset_mean_dice_NOT_FORMAL":         subset_dice_mean,
    "per_collection_mean_dice_NOT_FORMAL": per_collection_dice_mean,
    "per_patient":                         [{k: r[k] for k in _csv_fields}
                                            for r in per_patient],
    "outputs": {
        "montage":         montage_path,
        "csv":             csv_path,
        "json":            json_path,
        "per_patient_png": individual_paths,
    },
}
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_payload, f, indent=2)

# ── FINAL REPORT ───────────────────────────────────────────────────────
print()
print("=" * 72)
print("PER-COLLECTION VISUALIZATION SUMMARY")
print("=" * 72)
print(f"checkpoint                : {best_ckpt}")
print(f"epoch (human-readable)    : {ckpt_epoch_human}")
print(f"best_dice (self-reported) : {ckpt_best_dice:.4f}")
print(f"selection                 : {selection_mode}")
for note in selection_notes:
    print(f"  {note}")
print()
print("per-patient metrics:")
for r in per_patient:
    print(f"  [{r['collection']:5s}] {r['patient_id']:24s}  z={r['slice_index']:3d}  "
          f"Dice={r['dice']:.3f}  IoU={r['iou']:.3f}  Sens={r['sensitivity']:.3f}  "
          f"TP={r['tp_voxels']:>7d}  FP={r['fp_voxels']:>7d}  FN={r['fn_voxels']:>7d}")
print()
print("per-collection mean Dice (visualization subset — NOT formal):")
for coll in VIS_PC_COLLECTIONS:
    if coll in per_collection_dice_mean:
        n = len(per_collection_dice[coll])
        print(f"  {coll:8s} : {per_collection_dice_mean[coll]:.4f}   (n={n})")
    else:
        print(f"  {coll:8s} : (no cases)")
print()
print(f"overall subset mean Dice  : {subset_dice_mean:.4f}   "
      f"(n={len(per_patient)} — VISUALIZATION SUBSET, NOT the formal 306-case metric)")
print()
print("output files:")
print(f"  montage : {montage_path}")
print(f"  csv     : {csv_path}")
print(f"  json    : {json_path}")
for p in individual_paths:
    print(f"  case    : {p}")

# Display the montage inline in the notebook.
display(IPyImage(filename=montage_path))


In [ ]:
# ── 10. DOWNLOAD LINKS ────────────────────────────────────────────────
import os
from IPython.display import FileLink, display
for fname in ("best_model.pth", "last_checkpoint.pth",
              "train_log.json", "meta.json", "source_commit.txt"):
    p = os.path.join(OUTPUT_DIR_FULL, fname)
    if os.path.exists(p):
        print(fname); display(FileLink(p))
    else:
        print(f"(missing) {fname}")
